# Task 5 — Convolutional Neural Networks
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
- Custom CNN: 3 Conv blocks (Conv+BN+ReLU+MP), 2 FC layers, Dropout
- Data augmentation (horizontal flip, random crop, color jitter)
- Learning rate scheduler (ReduceLROnPlateau)
- Transfer learning: ResNet-50 frozen backbone then partial fine-tuning
- Filter visualisation, activation maps, sample predictions
- Dataset: CIFAR-10 (10 classes, 60k images, 32x32x3)

## Step 0 — Imports & Setup

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, MobileNet
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix


OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')

# print(f'TensorFlow version: 2.21.0')  # set by builder
print('TF OK — GPU:', len(tf.config.list_physical_devices('GPU')) > 0)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

TF OK — GPU: False
GPU available: False


## Step 1 — Data Loading & Preprocessing (CIFAR-10)

In [9]:
def prepare_cifar10_data():
    """Load CIFAR-10; normalise to [0,1]; return train/val/test split."""
    (x_train_full, y_train_full), (x_test, y_test) = \
        keras.datasets.cifar10.load_data()
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test       = x_test.astype('float32') / 255.0
    x_val   = x_train_full[-5000:];  y_val   = y_train_full[-5000:]
    x_train = x_train_full[:-5000];  y_train = y_train_full[:-5000]
    y_train = to_categorical(y_train, 10)
    y_val   = to_categorical(y_val,   10)
    y_test  = to_categorical(y_test,  10)
    print('Using CIFAR-10 dataset')
    print(f'Train: {x_train.shape}  Val: {x_val.shape}  Test: {x_test.shape}')
    print(f'Classes: {len(np.unique(np.argmax(y_train, axis=1)))}')
    return x_train, x_val, x_test, y_train, y_val, y_test

x_train, x_val, x_test, y_train, y_val, y_test = prepare_cifar10_data()
INPUT_SHAPE = x_train.shape[1:]   # (32, 32, 3)
NUM_CLASSES = y_train.shape[1]    # 10
CLASS_NAMES = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

Using CIFAR-10 dataset
Train: (45000, 32, 32, 3)  Val: (5000, 32, 32, 3)  Test: (10000, 32, 32, 3)
Classes: 10


## Step 2 — Custom CNN Architecture

```
Input(32x32x3) -> Conv(32,3x3) BN ReLU MaxPool(2x2)
             -> Conv(64,3x3) BN ReLU MaxPool(2x2)
             -> Conv(128,3x3) BN ReLU MaxPool(2x2)
             -> Flatten -> Dense(256) BN ReLU Dropout(0.5)
             -> Dense(128) BN ReLU Dropout(0.3)
             -> Dense(10, Softmax)
```

In [10]:
def build_custom_cnn(input_shape, num_classes):
    """3 Conv blocks + 2 FC layers; BN + Dropout."""
    return models.Sequential([
        layers.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Block 3
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Head
        layers.Flatten(),
        layers.Dense(256), layers.BatchNormalization(),
        layers.Activation('relu'), layers.Dropout(0.5),
        layers.Dense(128), layers.BatchNormalization(),
        layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ], name='custom_cnn')
print('Building custom CNN...')
model_custom = build_custom_cnn(INPUT_SHAPE, NUM_CLASSES)
model_custom.summary()

Building custom CNN...


Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 654,410 (2.50 MB)

 Trainable params: 653,194 (2.49 MB)

 Non-trainable params: 1,216 (4.75 KB)

## Step 3 — Training with Data Augmentation

In [11]:
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1)
datagen.fit(x_train)
lr_sched   = callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, verbose=1)
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)
model_custom.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                    loss='categorical_crossentropy', metrics=['accuracy'])
print('Training custom CNN (CIFAR-10, 30 epochs, augmented)...')
history_custom = model_custom.fit(
    datagen.flow(x_train, y_train, batch_size=128),
    validation_data=(x_val, y_val),
    epochs=10, callbacks=[lr_sched, early_stop], verbose=2)

Training custom CNN (CIFAR-10, 30 epochs, augmented)...
Epoch 1/10
352/352 - 36s - 103ms/step - accuracy: 0.4243 - loss: 1.6013 - val_accuracy: 0.2708 - val_loss: 2.0198 - learning_rate: 0.0010
Epoch 2/10
352/352 - 33s - 95ms/step - accuracy: 0.5575 - loss: 1.2372 - val_accuracy: 0.6194 - val_loss: 1.0588 - learning_rate: 0.0010
Epoch 3/10
352/352 - 33s - 93ms/step - accuracy: 0.6092 - loss: 1.1082 - val_accuracy: 0.6258 - val_loss: 1.0191 - learning_rate: 0.0010
Epoch 4/10
352/352 - 33s - 94ms/step - accuracy: 0.6436 - loss: 1.0113 - val_accuracy: 0.6624 - val_loss: 0.9662 - learning_rate: 0.0010
Epoch 5/10
352/352 - 33s - 93ms/step - accuracy: 0.6643 - loss: 0.9597 - val_accuracy: 0.6318 - val_loss: 1.1133 - learning_rate: 0.0010
Epoch 6/10
352/352 - 33s - 94ms/step - accuracy: 0.6815 - loss: 0.9086 - val_accuracy: 0.6460 - val_loss: 1.0653 - learning_rate: 0.0010
Epoch 7/10

Epoch 7: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
352/352 - 33s - 94ms/step - accur

## Step 4 — Training History (Accuracy + Loss)

In [12]:
ep = range(1, len(history_custom.history['accuracy'])+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, history_custom.history['accuracy'],     'b-', label='Train')
axes[0].plot(ep, history_custom.history['val_accuracy'], 'g--',label='Val')
axes[0].set_title('Custom CNN — Accuracy'); axes[0].legend()
axes[1].plot(ep, history_custom.history['loss'],     'b-', label='Train')
axes[1].plot(ep, history_custom.history['val_loss'], 'g--', label='Val')
axes[1].set_title('Custom CNN — Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_training_history.png'), dpi=150)
plt.close()
print('Saved T5_training_history.png')

Saved T5_training_history.png


## Step 5 — Custom CNN Evaluation & Confusion Matrix

In [13]:
test_loss_c, test_acc_c = model_custom.evaluate(x_test, y_test, verbose=0)
print(f'Custom CNN Test — Acc: {test_acc_c:.4f}  Loss: {test_loss_c:.4f}')
y_pred_c = np.argmax(model_custom.predict(x_test, verbose=0), axis=1)
y_true   = np.argmax(y_test, axis=1)
cm_c     = confusion_matrix(y_true, y_pred_c)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Custom CNN — Confusion Matrix  (Acc: {test_acc_c:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_custom_confusion_matrix.png'), dpi=150)
plt.close()
print('Saved T5_custom_confusion_matrix.png')
print(classification_report(y_true, y_pred_c, target_names=CLASS_NAMES))

Custom CNN Test — Acc: 0.7681  Loss: 0.6684
Saved T5_custom_confusion_matrix.png
              precision    recall  f1-score   support

    airplane       0.85      0.75      0.80      1000
  automobile       0.87      0.94      0.90      1000
        bird       0.75      0.61      0.67      1000
         cat       0.69      0.48      0.57      1000
        deer       0.74      0.72      0.73      1000
         dog       0.69      0.67      0.68      1000
        frog       0.60      0.96      0.74      1000
       horse       0.84      0.82      0.83      1000
        ship       0.83      0.91      0.87      1000
       truck       0.90      0.83      0.86      1000

    accuracy                           0.77     10000
   macro avg       0.78      0.77      0.76     10000
weighted avg       0.78      0.77      0.76     10000



## Step 6 — ResNet-50 Transfer Learning (Frozen Backbone)

In [14]:
base_rn = ResNet50(weights='imagenet', include_top=False, input_shape=(32,32,3))
base_rn.trainable = False
model_rn = models.Sequential([
    keras.Input(shape=(32,32,3)),
    base_rn,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), layers.BatchNormalization(),
    layers.Dense(10, activation='softmax'),
], name='resnet50_transfer')
model_rn.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
model_rn.summary()

Model: "resnet50_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,115,850 (91.99 MB)

 Trainable params: 527,626 (2.01 MB)

 Non-trainable params: 23,588,224 (89.98 MB)

## Step 7 — ResNet-50 Training & History

In [15]:
# Phase 1 — frozen backbone, head training (5 epochs)
print('Phase 1 — training head only (5 epochs)...')
base_rn.trainable = False
phase1 = model_rn.fit(
    x_train, y_train, batch_size=64,
    validation_data=(x_val, y_val), epochs=2, verbose=2)

# Phase 2 — partial fine-tuning, top 30 layers unfrozen (3 epochs, lr=1e-5)
print('Phase 2 — unfreezing top 30 layers, lr=1e-5 (3 epochs)...')
base_rn.trainable = True
for layer in base_rn.layers[:-30]:
    layer.trainable = False
model_rn.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])
phase2 = model_rn.fit(
    x_train, y_train, batch_size=64,
    validation_data=(x_val, y_val), epochs=3, verbose=2)

# Plot combined history
h  = phase1.history
h2 = phase2.history
ep1 = range(1, len(h['accuracy'])+1)
ep2 = range(len(h['accuracy'])+1, len(h['accuracy'])+len(h2['accuracy'])+1)
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
for ax, key in zip(ax2, ['accuracy', 'loss']):
    ax.plot(ep1, h[key],       'r-o', label='P1-Train')
    ax.plot(ep1, h['val_'+key], 'r--', label='P1-Val')
    ax.plot(ep2, h2[key],       'b-o', label='P2-Train')
    ax.plot(ep2, h2['val_'+key], 'b--', label='P2-Val')
    ax.set_title(f'ResNet-50 Transfer — {key.title()}'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_training_history.png'), dpi=150)
plt.close()
print('Saved T5_resnet50_training_history.png')

Phase 1 — training head only (5 epochs)...
Epoch 1/2
704/704 - 53s - 75ms/step - accuracy: 0.1435 - loss: 2.2769 - val_accuracy: 0.1396 - val_loss: 2.2789
Epoch 2/2
704/704 - 49s - 69ms/step - accuracy: 0.1297 - loss: 2.2541 - val_accuracy: 0.1714 - val_loss: 2.1962
Phase 2 — unfreezing top 30 layers, lr=1e-5 (3 epochs)...
Epoch 1/3
704/704 - 274s - 390ms/step - accuracy: 0.1875 - loss: 2.4176 - val_accuracy: 0.2656 - val_loss: 2.1808
Epoch 2/3
704/704 - 262s - 373ms/step - accuracy: 0.2604 - loss: 2.1601 - val_accuracy: 0.2830 - val_loss: 2.1947
Epoch 3/3
704/704 - 261s - 371ms/step - accuracy: 0.3077 - loss: 2.0005 - val_accuracy: 0.3212 - val_loss: 2.0591
Saved T5_resnet50_training_history.png


## Step 8 — ResNet-50 Evaluation & Confusion Matrix

In [16]:
test_loss_r, test_acc_r = model_rn.evaluate(x_test, y_test, verbose=0)
print(f'ResNet-50 Test — Acc: {test_acc_r:.4f}  Loss: {test_loss_r:.4f}')
y_pred_r = np.argmax(model_rn.predict(x_test, verbose=0), axis=1)
cm_r     = confusion_matrix(y_true, y_pred_r)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_r, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'ResNet-50 Transfer Learning  (Acc: {test_acc_r:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_confusion_matrix.png'), dpi=150)
plt.close()
print('Saved T5_resnet50_confusion_matrix.png')
print(classification_report(y_true, y_pred_r, target_names=CLASS_NAMES))
with open(os.path.join(OUTPUT_DIR, 'resnet50_metrics.txt'), 'w') as _fh:
    _fh.write(f'{test_acc_r:.4f}\n{test_loss_r:.4f}\n')

ResNet-50 Test — Acc: 0.3248  Loss: 2.0884
Saved T5_resnet50_confusion_matrix.png
              precision    recall  f1-score   support

    airplane       0.33      0.54      0.41      1000
  automobile       0.42      0.25      0.31      1000
        bird       0.24      0.35      0.28      1000
         cat       0.25      0.13      0.17      1000
        deer       0.29      0.16      0.21      1000
         dog       0.29      0.38      0.33      1000
        frog       0.41      0.14      0.21      1000
       horse       0.39      0.28      0.33      1000
        ship       0.37      0.46      0.41      1000
       truck       0.34      0.55      0.42      1000

    accuracy                           0.32     10000
   macro avg       0.33      0.32      0.31     10000
weighted avg       0.33      0.32      0.31     10000



## Step 9 — Learned Filter Visualisation

In [17]:
# First conv layer filters (R channel, 32 filters)
first_conv = model_custom.layers[0]
w  = first_conv.get_weights()[0]   # (3,3,3,32)
fig, axes = plt.subplots(4, 8, figsize=(22, 8))
axes = axes.flatten()
for i in range(32):
    f = w[:,:,0,i]
    f_norm = (f - f.min()) / (f.max() - f.min() + 1e-8)
    axes[i].imshow(f_norm, cmap='viridis'); axes[i].axis('off')
    axes[i].set_title(f'#{i}', fontsize=6)
plt.suptitle('Custom CNN — First Conv Layer Learned Filters (32/32)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_filters.png'), dpi=150)
plt.close()
print('Saved T5_cnn_filters.png')

Saved T5_cnn_filters.png


## Step 10 — Activation Map Visualisation

In [18]:
extractor = keras.Model(
    inputs=model_custom.inputs[0],
    outputs=[l.output for l in model_custom.layers
             if isinstance(l, (layers.Conv2D, layers.Activation))])
acts = extractor.predict(x_test[:3], verbose=0)
b1   = acts[4]    # Conv Block 1: (3, 16, 16, 32)
fig, axes = plt.subplots(3, 16, figsize=(24, 6))
for si in range(3):
    for ch in range(min(16, b1.shape[-1])):
        axes[si,ch].imshow(b1[si,:,:,ch], cmap='viridis')
        axes[si,ch].axis('off')
plt.suptitle('CNN Activation Maps — Conv Block 1 (3 images x 16 channels)', 
             fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_activations.png'), dpi=150)
plt.close()
print('Saved T5_cnn_activations.png')

Saved T5_cnn_activations.png


## Step 11 — Sample Image Predictions

In [19]:
np.random.seed(42)
idx = np.random.choice(len(x_test), 16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i, j in enumerate(idx):
    ax = axes.flatten()[i]
    ax.imshow(x_test[j])
    pred_l = CLASS_NAMES[y_pred_c[j]]
    true_l = CLASS_NAMES[y_true[j]]
    color  = 'green' if y_pred_c[j] == y_true[j] else 'red'
    ax.set_title(f'CNN: {pred_l}\nTrue: {true_l}', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle(
    'Custom CNN — CIFAR-10 Sample Predictions (green=correct, red=wrong)',
    fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_sample_predictions.png'), dpi=150)
plt.close()
print('Saved T5_sample_predictions.png')

Saved T5_sample_predictions.png


## Step 12 — Model Comparison: Custom CNN vs ResNet-50

In [20]:
models_list  = ['Custom CNN', 'ResNet-50\n(Transfer Learning)']
test_accs    = [test_acc_c, test_acc_r]
test_losses  = [test_loss_c, test_loss_r]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, vals, tit in zip(axes,
                          [test_accs, test_losses],
                          ['Test Accuracy', 'Test Loss']):
    b = ax.bar(models_list, vals,
               color=['#4472C4','#ED7D31'], edgecolor='black', width=0.5)
    ax.set_title(tit); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(b, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
               f'{val:.4f}', ha='center', fontweight='bold')
plt.suptitle('Task 5 — CNN Model Comparison on CIFAR-10',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_model_comparison.png'), dpi=150)
plt.close()
print(f'Custom CNN: acc={test_acc_c:.4f}  loss={test_loss_c:.4f}')
print(f'ResNet-50 : acc={test_acc_r:.4f}  loss={test_loss_r:.4f}')
print('Saved T5_model_comparison.png')

Custom CNN: acc=0.7681  loss=0.6684
ResNet-50 : acc=0.3248  loss=2.0884
Saved T5_model_comparison.png


## Step 13 — Results Discussion

In [21]:
print('\n=== T5 RESULTS SUMMARY ===')
print(f'Custom CNN (from scratch): acc={test_acc_c:.4f}')
print(f'ResNet-50 (transfer):      acc={test_acc_r:.4f}')
print('')
print('Key take-aways:')
print('- Augmentation: rot, flip, shift, zoom — active in training.')
print('- Filters show edge / texture detectors.')
print('- Activation maps highlight channel-specific feature detection.')
print('- ResNet-50 needs full fine-tuning >>30 epochs at lr=1e-5 on 32x32.')
print('- Next: EfficientNet-B0, TTA, warm-up + cosine schedule.')


=== T5 RESULTS SUMMARY ===
Custom CNN (from scratch): acc=0.7681
ResNet-50 (transfer):      acc=0.3248

Key take-aways:
- Augmentation: rot, flip, shift, zoom — active in training.
- Filters show edge / texture detectors.
- Activation maps highlight channel-specific feature detection.
- ResNet-50 needs full fine-tuning >>30 epochs at lr=1e-5 on 32x32.
- Next: EfficientNet-B0, TTA, warm-up + cosine schedule.
